# CAPE-TTA × OpenVLA × LIBERO — Free-Colab Oracle Pilot

This notebook runs a **real OpenVLA checkpoint** on LIBERO and evaluates K counterfactual first-action candidates with CAPE-TTA's validated fresh-environment prefix-replay oracle.

**Pilot only:** the run uses 8-bit OpenVLA + FP16 on a free T4-class GPU to fit memory. Official LIBERO sparse success and CAPE's simulation-only soft task potential are logged separately. Do not report this pilot as the final paper benchmark.

## Run
Select **Runtime → Change runtime type → T4 GPU → Save** (代码执行程序 → 更改运行时类型 → T4 GPU → 保存). Then run the single code cell below. Each stage must finish before the next starts. GPU allocation still depends on Colab availability.

If a stage fails, the cell stops and downloads `cape_colab_failure.json`. Save this notebook with outputs to GitHub for diagnosis. Existing checkout/results are preserved on rerun; to use a newer repository revision, start a fresh Colab runtime.


In [5]:
# Run this one cell: GPU → environment → repositories → preflight → pilot.
import json, traceback
from pathlib import Path
STAGES = [
  {
    "name": "GPU check",
    "source": "import subprocess, shutil, sys, os\nsmi = shutil.which('nvidia-smi')\nif smi is None:\n    raise RuntimeError('GPU unavailable. Choose Runtime / 代码执行程序 → Change runtime type / 更改运行时类型 → T4 GPU → Save, reconnect, then run this cell.')\nsubprocess.run([smi, '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)\nprint('Host Python:', sys.version)\n"
  },
  {
    "name": "Environment setup",
    "source": "import os, subprocess, textwrap\ndef sh(cmd):\n    print('\\n$', cmd)\n    with subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:\n        tail = []\n        for line in proc.stdout:\n            print(line, end='', flush=True)\n            tail.append(line)\n            tail = tail[-100:]\n        status = proc.wait()\n        if status:\n            raise subprocess.CalledProcessError(status, cmd, output=''.join(tail))\n\nsh('apt-get update -qq')\nsh('DEBIAN_FRONTEND=noninteractive apt-get install -y -qq curl git ffmpeg libosmesa6-dev libgl1 libglx-mesa0 libglfw3 build-essential')\nif not os.path.exists('/content/micromamba'):\n    sh(\"cd /content && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba && mv bin/micromamba /content/micromamba\")\nif not os.path.exists('/content/cape-env/bin/python'):\n    sh('/content/micromamba create -y -p /content/cape-env -c conda-forge python=3.10 pip')\nPY='/content/cape-env/bin/python'\nPIP=f'{PY} -m pip'\nsh(f'{PIP} install --upgrade pip')\nsh(f'{PIP} install --index-url https://download.pytorch.org/whl/cu121 torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0')\nsh(f'''{PIP} install \\\n  numpy==1.26.4 scipy==1.11.4 matplotlib==3.7.5 \\\n  transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 accelerate==0.29.3 \\\n  bitsandbytes==0.43.1 peft==0.11.1 sentencepiece==0.1.99 einops rich protobuf \\\n  tensorflow==2.15.0 pillow==10.3.0 safetensors huggingface_hub \\\n  'imageio[ffmpeg]' robosuite==1.4.1 bddl easydict cloudpickle gym==0.25.2''')\nprint('Environment ready:', PY)\n"
  },
  {
    "name": "Repository setup",
    "source": "import os, subprocess\ndef sh(cmd):\n    print('\\n$', cmd)\n    with subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:\n        tail = []\n        for line in proc.stdout:\n            print(line, end='', flush=True)\n            tail.append(line)\n            tail = tail[-100:]\n        status = proc.wait()\n        if status:\n            raise subprocess.CalledProcessError(status, cmd, output=''.join(tail))\nPY='/content/cape-env/bin/python'\nPIP=f'{PY} -m pip'\nfrom pathlib import Path\nfor repo_url, destination in [\n    ('https://github.com/weichezhang/CAPE-TTA.git', '/content/CAPE-TTA'),\n    ('https://github.com/Lifelong-Robot-Learning/LIBERO.git', '/content/LIBERO'),\n]:\n    if not Path(destination).exists():\n        subprocess.run(['git', 'clone', '--depth', '1', repo_url, destination], check=True)\n    elif not Path(destination, '.git').is_dir():\n        raise RuntimeError(f'{destination} exists but is not a Git checkout; refusing to overwrite it.')\n    else:\n        print('Reusing checkout:', destination)\n        subprocess.run(['git', '-C', destination, 'status', '--short'], check=True)\nsh(f'{PIP} install -e /content/CAPE-TTA --no-deps')\nsh(f'{PIP} install -e /content/LIBERO --no-deps')\nos.makedirs(os.path.expanduser('~/.libero'), exist_ok=True)\ncfg='''benchmark_root: /content/LIBERO/libero/libero\\nbddl_files: /content/LIBERO/libero/libero/bddl_files\\ninit_states: /content/LIBERO/libero/libero/init_files\\ndatasets: /content/LIBERO/libero/datasets\\nassets: /content/LIBERO/libero/libero/assets\\n'''\nopen(os.path.expanduser('~/.libero/config.yaml'),'w').write(cfg)\ncommit=subprocess.check_output(['git','-C','/content/CAPE-TTA','rev-parse','HEAD'], text=True).strip()\nprint('CAPE-TTA commit:', commit)\n"
  },
  {
    "name": "Runtime preflight",
    "source": "import subprocess, textwrap\nPY='/content/cape-env/bin/python'\ncode=r'''import torch, transformers, tokenizers, timm, bitsandbytes as bnb, numpy as np\nprint('torch=', torch.__version__, 'cuda=', torch.version.cuda)\nprint('gpu=', torch.cuda.get_device_name(0), 'capability=', torch.cuda.get_device_capability(0))\nprint('transformers=', transformers.__version__, 'tokenizers=', tokenizers.__version__, 'timm=', timm.__version__, 'numpy=', np.__version__, 'bnb=', bnb.__version__)\nassert torch.cuda.is_available()\n'''\nsubprocess.run([PY,'-c',code], check=True)\n"
  },
  {
    "name": "OpenVLA Oracle pilot",
    "source": "import os, subprocess, json, pathlib\nPY='/content/cape-env/bin/python'\ncommit=subprocess.check_output(['git','-C','/content/CAPE-TTA','rev-parse','HEAD'], text=True).strip()\nenv=os.environ.copy()\nenv.update({\n    'MUJOCO_GL':'osmesa',\n    'PYOPENGL_PLATFORM':'osmesa',\n    'LIBERO_REPO_ROOT':'/content/LIBERO',\n    'CAPE_COMMIT':commit,\n    'TOKENIZERS_PARALLELISM':'false',\n    'TF_CPP_MIN_LOG_LEVEL':'2',\n    'BITSANDBYTES_NOWELCOME':'1',\n    'HF_HOME':'/content/hf_cache',\n})\nout='/content/CAPE-TTA/results/openvla_oracle_pilot_colab.json'\ncmd=[PY,'/content/CAPE-TTA/scripts/run_openvla_oracle_pilot.py',\n     '--checkpoint','openvla/openvla-7b-finetuned-libero-spatial',\n     '--suite','libero_spatial','--task-id','0','--init-state-id','0',\n     '--candidates','4','--horizon','12','--settle-steps','10',\n     '--temperature','0.7','--top-p','0.95','--seed','7',\n     '--no-flash-attention','--load-in-8bit','--input-dtype','fp16',\n     '--output',out]\nprint('Running:', ' '.join(cmd))\nsubprocess.run(cmd, env=env, check=True)\nresult=json.load(open(out))\nprint('\\n=== PILOT SUMMARY ===')\nprint('GPU:', result['runtime']['gpu'])\nprint('wrapper parity max abs:', result['wrapper_parity_max_abs'])\nprint('fresh replay:', result['fresh_replay_determinism'])\nprint('base soft gain:', result['base_soft_gain'])\nprint('oracle best index:', result['oracle_best_index'])\nprint('oracle best soft gain:', result['oracle_best_soft_gain'])\nprint('oracle margin over base:', result['oracle_margin_over_base'])\n"
  }
]
for stage in STAGES:
    print('\n=== ' + stage['name'] + ' ===', flush=True)
    try:
        exec(compile(stage['source'], stage['name'], 'exec'), globals())
    except Exception as exc:
        diagnostic = {
            'status': 'failed',
            'stage': stage['name'],
            'error_type': type(exc).__name__,
            'error': str(exc),
            'command_output_tail': getattr(exc, 'output', None),
            'traceback': traceback.format_exc(),
        }
        diagnostic_path = Path('/content/cape_colab_failure.json')
        diagnostic_path.write_text(json.dumps(diagnostic, indent=2), encoding='utf-8')
        print('Stopped at: ' + stage['name'])
        print('Diagnostic saved to: ' + str(diagnostic_path))
        try:
            from google.colab import files
            files.download(str(diagnostic_path))
        except Exception:
            pass
        raise
    print('Completed: ' + stage['name'], flush=True)
print('\nPilot finished; downloading the raw result.')
from google.colab import files
files.download('/content/CAPE-TTA/results/openvla_oracle_pilot_colab.json')



=== GPU check ===
Host Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Completed: GPU check

=== Environment setup ===

$ apt-get update -qq
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)

$ DEBIAN_FRONTEND=noninteractive apt-get install -y -qq curl git ffmpeg libosmesa6-dev libgl1 libglx-mesa0 libglfw3 build-essential

$ /content/cape-env/bin/python -m pip install --upgrade pip

$ /content/cape-env/bin/python -m pip install --index-url https://download.pytorch.org/whl/cu121 torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0
Looking in indexes: https://download.pytorch.org/whl/cu121

$ /content/cape-env/bin/python -m pip install   numpy==1.26.4 scipy==1.11.4 matplotlib==3.7.5   transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 accelerate==0.29.3   bitsandbytes==0.43.1 peft==0.11.1 sentencepiece==0.1.99 einops rich protobuf   tensor

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CalledProcessError: Command '['/content/cape-env/bin/python', '/content/CAPE-TTA/scripts/run_openvla_oracle_pilot.py', '--checkpoint', 'openvla/openvla-7b-finetuned-libero-spatial', '--suite', 'libero_spatial', '--task-id', '0', '--init-state-id', '0', '--candidates', '4', '--horizon', '12', '--settle-steps', '10', '--temperature', '0.7', '--top-p', '0.95', '--seed', '7', '--no-flash-attention', '--load-in-8bit', '--input-dtype', 'fp16', '--output', '/content/CAPE-TTA/results/openvla_oracle_pilot_colab.json']' returned non-zero exit status 1.

### What to do with the JSON
Upload `openvla_oracle_pilot_colab.json` back to the CAPE-TTA ChatGPT project. The next stage is to verify candidate diversity and oracle margin, then scale to more init states/tasks and fit the conformal calibration split. Do **not** treat this single pilot as the paper's final benchmark result.
